# Enforcement that doesn't know what tool it's looking at

By the end of notebook 9 the system had three ways to stop a bad tool call, and
all three share a blind spot.

| Layer | Knows about | Blind to |
| --- | --- | --- |
| the agent's `tools` list | which capabilities exist | anything at call time |
| schema validation | one tool's argument types | the case, the caller |
| semantic validation | one tool's arguments *and* the case | who is calling, and every other tool |

Read the third row again. `validate_refund` is an expert on refunds. It has no
opinion about `create_return_authorization`, and it has never heard of a tool
added next week. So a rule like **"no sub-agent may ever write to disk"** cannot
be written anywhere — it has to be re-stated inside every write tool's validator,
and stay true in every spoke's tool list, forever.

Notebook 9 claimed that property was already true. It was, but only by
coincidence: no spoke's tool list happened to contain a write tool. Nothing
enforced it. Add a fourth sub-agent carelessly and the guarantee evaporates with
no error, no test failure, and no sign anything changed.

A **hook** is the missing layer. It runs on every tool call, knows the *caller*
and the *case*, and knows nothing about which tool it's looking at:

- **`PreToolUse`** runs before execution and can **allow**, **deny**, or
  **modify** the input.
- **`PostToolUse`** runs after and can **record** or **rewrite** the result
  before it reaches any model's context.

That inverts the question from "is this call well-formed?" to "is this caller
allowed to do this, right now, on this case?" — and one rule answers it for every
tool that exists and every tool you add later.

## Notebook 9 is now a library too

The same promotion as last time. The coordinator, the three spokes, `Task`,
`report_finding` and `run_task` moved into **`shop/agents.py`**, so this notebook
can be about enforcement rather than re-explaining delegation:

| Import | What it is | Taught in |
| --- | --- | --- |
| `coordinator_spec(model)` | the hub: Task + the three write tools | nb 9 |
| `SPOKE_TOOLS`, `spoke_spec` | the three read-only sub-agents | nb 9 |
| `TASK_TOOL`, `REPORT_FINDING` | delegation and the handoff contract | nb 9 |
| `run_task`, `build_brief` | isolated sub-agent runs, explicit context | nb 9 |
| `send_message` | one customer turn, coordinator + spokes | nb 9 |

Notebook 9 still defines all of this in its own cells, because there it is the
subject. Treat `shop/agents.py` as the canonical copy.

**One genuinely new parameter: `wrap`.** Every dispatcher `shop/agents.py` builds
accepts `wrap(dispatch, agent_name)` and applies it to sub-agents as well as to
the coordinator. Without that, an interposer would only ever see the hub's calls
— and "no sub-agent may write" would be unenforceable at exactly the layer that
matters. This is the seam notebook 9 promised and left unused.

**And one piece of housekeeping worth explaining rather than hiding.** The
notebook 9 transcript ended with `billing_analysis` reporting that A1006 "has
already been fully refunded" — because `refunds.json` still held a refund from an
earlier, buggy run. Those files are *inputs* to the fact tools as well as outputs
of the action tools, so leftover state doesn't crash anything; it quietly changes
the answer. `reset_records()` clears them, and the setup cell calls it.

In [1]:
import json
import re

from dotenv import load_dotenv
from anthropic import Anthropic

from shop import ACTION_TOOLS, execute_tool, is_closed, new_case
from shop.agents import CASE_LOCK, TASK_LOG, send_message
from shop.data import payment_events, reset_records

load_dotenv()

client = Anthropic()

COORDINATOR_MODEL = "claude-sonnet-5"
SPOKE_MODEL = "claude-haiku-4-5"

# Clear returns.json / refunds.json / escalations.json. See the note above: these
# are read back by the fact tools, so yesterday's run is today's evidence.
print("cleared:", reset_records() or "nothing to clear")

case = new_case()
messages = []

cleared: ['returns.json', 'refunds.json', 'escalations.json']


## The contract

A hook is a function over one tool call. `PreToolUse` hooks return one of three
things:

```
None                        allow — this hook has no opinion
deny(reason, rule=...)      refuse; the tool never runs
modify(input, rule=...)     rewrite the arguments, then carry on
```

`PostToolUse` hooks receive the result and return either `None` (leave it alone)
or a replacement `(result, is_error)`.

Three details in the engine below are worth more than the rest of it.

**A denial is a `tool_result` with `is_error: True`, not an exception.** The
model has to *learn* it was blocked, in the same channel it learns everything
else, or it will keep trying. The refusal carries the reason and the rule name,
so the next thing the coordinator does is informed rather than a retry.

**The audit is written by the engine, not by a hook.** It records allows, denies
and modifications alike. An audit trail you can forget to install is not an audit
trail, and making it a hook would make forgetting it possible.

**The audit is load-bearing, not decoration.** `no_repeat_calls` below answers
"have we done this already?" by reading `case["audit"]`. Once the record is the
thing enforcement consults, it cannot quietly rot — a broken audit becomes a
broken system rather than a boring file nobody reads.

In [2]:
def deny(reason: str, *, rule: str) -> dict:
    """Refuse the call. The tool does not run."""
    return {"action": "deny", "reason": reason, "rule": rule}


def modify(new_input: dict, *, rule: str, note: str) -> dict:
    """Rewrite the arguments and carry on."""
    return {"action": "modify", "input": new_input, "rule": rule, "note": note}


PRE_HOOKS: list = []
POST_HOOKS: list = []


def _register(registry, fn):
    # Replace by qualified name so re-running a cell re-registers rather than
    # stacking a second copy of the same rule.
    key = f"{fn.__module__}.{fn.__qualname__}"
    for i, existing in enumerate(registry):
        if f"{existing.__module__}.{existing.__qualname__}" == key:
            registry[i] = fn
            return fn

    registry.append(fn)
    return fn


def pre_tool_use(fn):
    """Runs before the tool. Returns None, deny(...) or modify(...)."""
    return _register(PRE_HOOKS, fn)


def post_tool_use(fn):
    """Runs after the tool. Returns None or a replacement (result, is_error)."""
    return _register(POST_HOOKS, fn)


def fingerprint(call: dict) -> str:
    """Identity of a call: who, what, with exactly which arguments."""
    return f"{call['agent']}:{call['tool']}:{json.dumps(call['input'], sort_keys=True)}"


def record(case: dict, call: dict, decision: str, detail: str = "") -> None:
    """Append to the case audit. Locked — parallel sub-agents write here too."""
    with CASE_LOCK:
        case["audit"].append({
            "agent": call["agent"],
            "tool": call["tool"],
            "input": call["input"],
            "decision": decision,
            "detail": detail,
            "fingerprint": fingerprint(call),
        })


def hooked(inner, agent: str):
    """Wrap a dispatcher so every call through it passes the hooks.

    `agent` is closed over rather than passed in, because the caller's identity
    must not be something the caller can state. A sub-agent cannot claim to be
    the coordinator: it never gets to say who it is.
    """

    def dispatch(tool_name, tool_input, case):
        call = {"agent": agent, "tool": tool_name, "input": tool_input}

        for hook in PRE_HOOKS:
            decision = hook(call, case)
            if decision is None:
                continue

            if decision["action"] == "deny":
                record(case, call, "deny", decision["rule"])
                print(f"[hook] DENIED {agent} → {tool_name} ({decision['rule']})")
                # A tool_result, not an exception: the model has to learn it was
                # blocked, in the channel it learns everything else.
                return json.dumps({
                    "ok": False,
                    "error": decision["reason"],
                    "blocked_by": decision["rule"],
                }), True

            if decision["action"] == "modify":
                call["input"] = decision["input"]
                record(case, call, "modify", f"{decision['rule']}: {decision['note']}")
                print(f"[hook] rewrote {agent} → {tool_name} ({decision['rule']})")

        result, is_error = inner(call["tool"], call["input"], case)

        notes = []
        for hook in POST_HOOKS:
            replacement = hook(call, result, is_error, case)
            if replacement is not None:
                result, is_error = replacement
                notes.append(hook.__name__)

        record(case, call, "allow", ", ".join(notes))
        return result, is_error

    return dispatch

## PreToolUse: the rules that were previously true by accident

Four rules. None of them names a specific tool — they work in categories, which
is what lets one line cover tools that don't exist yet.

**`only_the_coordinator_may_act`** is the notebook-9 debt paid off. A sub-agent
calling any action tool, or calling `Task`, is denied. Previously this held
because every spoke's tool list omitted those names; now it holds because a rule
says so. Add a fourth spoke and paste the wrong tool list into it and the rule
still catches it — a guarantee that survives your future mistakes is worth more
than one that depends on you not making them.

Note where the caller's identity comes from: `hooked(inner, agent)` closes over
it. A sub-agent never gets to state who it is, so it cannot claim to be the
coordinator. Identity that travels in the arguments is identity the caller
controls.

**`freeze_closed_case`** stops all writes and all delegation once the case is
terminal. Notebook 8 had this as one clause inside each action validator;
notebook 9 inherited it. Here it is one rule for every mutating tool, and unlike
the validator version it also covers `Task` — spending money on a sub-agent for
a case that's already finished is waste, not danger, but it's still wrong.

**`no_repeat_calls`** denies a call this agent has already made with exactly
these arguments. This is the loop-suppressor: notebook 9's coordinator re-tasked
`order_investigation` twice for the same order, and a model that gets an
unsatisfying result will often try the identical call again. The result is
already in its context; the second call is pure cost.

**`delegation_budget`** caps sub-agent runs per case. Every `Task` is a whole
model conversation, so the coordinator's ability to fan out is also its ability
to spend without limit.

**`normalize_order_ids`** is the `modify` case, and it turns out to matter more
than it looks. It rewrites `" a1006 "` to `"A1006"` wherever an order ID appears,
including inside `Task`'s nested `context`. The tools were already
case-insensitive, so on its own this is cosmetic — but the audit entry is written
*after* the rewrite, which means `no_repeat_calls` compares canonical arguments.
Without it, `" a1006 "` and `"A1006"` are two different calls and the duplicate
slips through. Normalising before you fingerprint is what makes an
identity-based rule work at all.

Be careful with the mechanism, though: a `PreToolUse` hook can silently rewrite
any argument of any call. Rewriting a *format* is housekeeping. Rewriting a
*decision* — bumping an urgency, changing an amount — would mean the audit shows
a call the model never made.

### Two things about ordering

**Hooks run after validation here.** `dispatch` is called by `run_agent` only
once the schema and semantic layers have passed, so a `PreToolUse` hook never
sees a call the validators already refused. The practical consequence: `modify`
canonicalises a call that is *already legal*; it cannot rescue one that isn't.
If you want the opposite, the seam moves into `run_agent` ahead of
`validate_tool_call` — worth knowing which you've built, because "the hook will
clean it up" is false in this arrangement.

**And in a correctly configured system, most of these denials never fire.** A
spoke that doesn't have `issue_refund` in its tool list gets stopped one layer
earlier, by the loop, with "unknown tool". That is not an argument against the
hook — it's the whole point. The hook is what holds when the tool list is
*wrong*, which is the failure you won't notice any other way.

In [3]:
# Tools that change something. Note this is a *category*, not a list of names
# repeated in each rule — a new write tool joins ACTION_TOOLS and every rule
# below covers it with no edit.
MUTATING = ACTION_TOOLS | {"Task"}

MAX_TASKS_PER_CASE = 6


@pre_tool_use
def only_the_coordinator_may_act(call: dict, case: dict):
    """The whole hub-and-spoke security claim, as one rule.

    In notebook 9 this was true because every spoke's tool list happened to omit
    the write tools. True-by-omission is not enforcement: nothing would have
    complained if a fourth sub-agent had been given issue_refund by mistake.
    """
    if call["agent"] == "coordinator" or call["tool"] not in MUTATING:
        return None

    return deny(
        f"'{call['agent']}' is a read-only sub-agent and may not call "
        f"'{call['tool']}'. Report what you found with report_finding and let the "
        "coordinator decide what to do about it.",
        rule="only_the_coordinator_may_act",
    )


@pre_tool_use
def freeze_closed_case(call: dict, case: dict):
    """Once a case is resolved, nothing may change it — or spend on it."""
    if not is_closed(case) or call["tool"] not in MUTATING:
        return None

    return deny(
        f"This case is already '{case['state']}' and is closed. No further actions "
        "or investigations may be taken on it.",
        rule="freeze_closed_case",
    )


@pre_tool_use
def no_repeat_calls(call: dict, case: dict):
    """Deny a call this agent already made with exactly these arguments.

    Answered from case["audit"] — which is what makes the audit trail load-bearing
    rather than decorative. A rejected call doesn't count, so a genuine retry
    after a validation failure still gets through.
    """
    already = any(
        entry["fingerprint"] == fingerprint(call) and entry["decision"] == "allow"
        for entry in case["audit"]
    )
    if not already:
        return None

    return deny(
        f"You already called '{call['tool']}' with exactly these arguments on this "
        "case, and the result is in your context. Read it again rather than "
        "repeating the call; if you need something different, change the arguments.",
        rule="no_repeat_calls",
    )


@pre_tool_use
def delegation_budget(call: dict, case: dict):
    """Cap sub-agent runs per case. Every Task is a whole model conversation."""
    if call["tool"] != "Task":
        return None

    used = sum(
        1 for entry in case["audit"]
        if entry["tool"] == "Task" and entry["decision"] == "allow"
    )
    if used < MAX_TASKS_PER_CASE:
        return None

    return deny(
        f"Delegation budget spent: {used} sub-agent tasks have already run on this "
        "case. Decide with the findings you have, or escalate to a human.",
        rule="delegation_budget",
    )


_ORDER_ID_KEY = "order_id"


@pre_tool_use
def normalize_order_ids(call: dict, case: dict):
    """MODIFY: canonicalise every order_id in the arguments, however deeply nested."""

    def clean(value):
        if isinstance(value, dict):
            return {
                key: (
                    inner.strip().upper()
                    if key == _ORDER_ID_KEY and isinstance(inner, str)
                    else clean(inner)
                )
                for key, inner in value.items()
            }
        return value

    cleaned = clean(call["input"])
    if cleaned == call["input"]:
        return None

    return modify(
        cleaned,
        rule="normalize_order_ids",
        note="canonicalised order IDs to uppercase, trimmed",
    )

## PostToolUse: rewriting what the model is allowed to see

A `PostToolUse` hook sits between the tool and the context. Whatever it returns
is what gets appended to `messages` — so it is the last place data can be
stopped before it becomes permanent.

`redact_card_numbers` is the case that makes this concrete. `payments.json`
carries full card numbers, and `get_payment_events` returns the ledger verbatim.
Without a hook, the first sub-agent to look at a payment puts a PAN into a
conversation that then gets forwarded to the coordinator, written into a finding,
and attached to an escalation ticket. Nothing malicious happens; it just spreads,
and every copy is permanent.

Redaction at this layer works differently from asking nicely:

- It happens **before** the model sees the data, so there is nothing to leak.
- It applies to **every tool**, including ones added later that nobody thought to
  check.
- It cannot be talked out of it, because it isn't a participant in the
  conversation.

This is the strongest argument for hooks generally. A validator can refuse an
action; only a `PostToolUse` hook can change what is true about the context
itself.

In [4]:
# Test-card PANs in payments.json — 13 to 16 digits, no separators.
_PAN_RE = re.compile(r"\b\d{13,16}\b")


@post_tool_use
def redact_card_numbers(call: dict, result: str, is_error: bool, case: dict):
    """Mask card numbers before the result becomes part of any conversation.

    Runs on every tool's output, not just the payment ones. A rule that only
    covers the tools you remembered is a rule that fails on the tool you add next
    month.
    """
    masked = _PAN_RE.sub(lambda m: f"****{m.group()[-4:]}", result)
    if masked == result:
        return None

    return masked, is_error


@post_tool_use
def note_failed_writes(call: dict, result: str, is_error: bool, case: dict):
    """Make a failed write loud in the log. Changes nothing the model sees.

    A PostToolUse hook that returns None is still useful: observation without
    interference is most of what you want most of the time.
    """
    if is_error and call["tool"] in ACTION_TOOLS:
        print(f"[hook] failed write: {call['agent']} → {call['tool']}")

    return None

## Wiring

One argument. `wrap=hooked` is threaded to the coordinator's dispatcher *and*,
inside `run_task`, to every sub-agent's — so a hook sees every tool call in the
system regardless of who made it.

Nothing else changes. The loop doesn't know hooks exist, `shop/agents.py`
doesn't know what `hooked` does, and the tools are untouched. That is the point
of putting enforcement at a seam rather than inside the things being enforced.

In [5]:
def reply_to(text: str) -> str:
    """One customer turn, with every tool call in the system passing the hooks."""
    messages.append({"role": "user", "content": text})
    return send_message(
        client, messages, case,
        coordinator_model=COORDINATOR_MODEL,
        spoke_model=SPOKE_MODEL,
        wrap=hooked,
    )

## Proving it without the model

Hooks are ordinary functions over ordinary arguments, which means the enforcement
can be tested with no model in the loop at all — deterministically, in
milliseconds, in CI.

That is worth pausing on. Everything protecting this system before now was
either a prompt (untestable), a tool list (testable only by inspection), or a
validator (testable, but only one tool at a time). The security property "no
sub-agent can write" is now a single assertion.

The cell below wraps `execute_tool` directly, bypassing every model, and checks
that each rule fires.

In [6]:
demo = new_case()
as_spoke = hooked(execute_tool, "billing_analysis")
as_coordinator = hooked(execute_tool, "coordinator")

print("1. sub-agent tries to move money")
result, is_error = as_spoke("issue_refund", {"order_id": "A1006", "amount": 149.0}, demo)
print("   ", result, "\n")

print("2. sub-agent tries to delegate")
result, is_error = as_spoke("Task", {"agent": "policy_review"}, demo)
print("   ", json.loads(result)["blocked_by"], "\n")

print("3. coordinator reads payments — order_id rewritten, card numbers masked")
ledger, is_error = as_coordinator("get_payment_events", {"order_id": " a1006 "}, demo)
print("   ", ledger[:230], "...\n")

print("4. the same call again")
result, is_error = as_coordinator("get_payment_events", {"order_id": "A1006"}, demo)
print("   ", json.loads(result)["blocked_by"], "\n")

print("5. writes after the case closes")
demo["state"] = "escalated"
result, is_error = as_coordinator(
    "create_return_authorization",
    {"order_id": "A1001", "reason": "defective_item", "note": "faulty"},
    demo,
)
print("   ", json.loads(result)["blocked_by"], "\n")

# The security properties, as assertions. No model was involved in any of this.
assert not any(a["decision"] == "allow" for a in demo["audit"] if a["agent"] != "coordinator")
assert not re.search(r"\b\d{13,16}\b", ledger), "a card number reached the context"
assert "378282246310005" in json.dumps(payment_events("A1006")), "the raw PAN is still on disk"

print("audit:", [(a["agent"], a["tool"], a["decision"]) for a in demo["audit"]])
print("\nno sub-agent call was ever allowed; no PAN ever reached a conversation")

1. sub-agent tries to move money
[hook] DENIED billing_analysis → issue_refund (only_the_coordinator_may_act)
    {"ok": false, "error": "'billing_analysis' is a read-only sub-agent and may not call 'issue_refund'. Report what you found with report_finding and let the coordinator decide what to do about it.", "blocked_by": "only_the_coordinator_may_act"} 

2. sub-agent tries to delegate
[hook] DENIED billing_analysis → Task (only_the_coordinator_may_act)
    only_the_coordinator_may_act 

3. coordinator reads payments — order_id rewritten, card numbers masked
[hook] rewrote coordinator → get_payment_events (normalize_order_ids)
    {"order_id": "A1006", "order_total": 149.0, "events": [{"order_id": "A1006", "date": "2026-07-20", "type": "authorized", "amount": 149.0, "method": "amex ****0005", "reference": "PAY-A1006-01", "card_number": "****0005"}, {"order_ ...

4. the same call again
[hook] DENIED coordinator → get_payment_events (no_repeat_calls)
    no_repeat_calls 

5. writes afte

## Chat

The scenarios from notebook 9 still apply. What's worth watching now is the
`[hook]` lines interleaved with the tool log — that's enforcement happening,
visible, on calls nobody wrote a validator for.

- **"A1006 never turned up and I think you charged me twice"** — watch
  `normalize_order_ids` and `redact_card_numbers` fire on the billing sub-agent's
  ledger read. The PAN never enters any conversation.
- **"A1001 arrived defective, I'd like my money back"** — the happy path, and a
  chance to see `no_repeat_calls` catch the coordinator re-reading an order it
  already looked up.
- **Keep talking after the case closes** — the chat loop breaks on a closed case,
  so to see `freeze_closed_case` fire you'll want the direct test above, or to
  comment out the break.

If you want a rule to *stop* applying, comment out its decorator and re-run the
cell — `_register` replaces by name, so re-running never stacks duplicates, but
it also won't remove a hook you've deleted. Restart the kernel for that.

In [7]:
while True:
    try:
        user_input = input("You: ")
        print(f"User: {user_input}")
    except (EOFError, KeyboardInterrupt):
        break

    if not user_input.strip() or user_input.lower() in ("quit", "exit"):
        break

    print(f"Assistant: {reply_to(user_input)}")

    if is_closed(case):
        print(f"\n--- case closed ({case['state']}): {case['actions']} ---")
        break

User: Hello
Assistant: Hi there! How can I help you today? If this is about an order, feel free to share the order number.
User: Yes, I have a problem with A1009
Assistant: I'd like to hear a bit more first — what's going on with order A1009? (Missing item, want to return it, billing issue, etc.)
User: It was a week and back created charge back for me for that order
[coordinator] → Task({'agent': 'order_investigation', 'objective': 'Look up order A1009 and report its status, order date, item, total, and delivery/carrier history.', 'context': {'order_id': 'A1009', 'customer_statement': 'It was a week and back created charge back for me for that order', 'notes': 'Customer claims their bank initiated a chargeback for this order.'}, 'include_findings': []})
[coordinator] → Task({'agent': 'billing_analysis', 'objective': 'Check order A1009 for any charges, refunds, or disputes/chargebacks on record.', 'context': {'order_id': 'A1009', 'customer_statement': 'It was a week and back created cha

## The audit trail

Every tool call attempted on this case: who made it, what they asked for, and
what the system decided. Denials included — which is the half that was missing
before, and usually the half you want when something has gone wrong.

This is also the last leg of the structured handoff. `escalate_to_human` now
attaches the denied attempts to the ticket alongside the findings, so a human
picking up the case sees not only what the machine established but what it tried
to do and was stopped from doing. "The assistant attempted to refund a disputed
charge and was blocked" is frequently the single most important line on a support
ticket, and it is exactly the line that no system built out of prompts can
produce.

In [8]:
print("=== audit ===")
for entry in case["audit"]:
    mark = {"allow": "ok  ", "deny": "DENY", "modify": "mod "}[entry["decision"]]
    detail = f"  ({entry['detail']})" if entry["detail"] else ""
    print(f"{mark} {entry['agent']:<20} {entry['tool']:<28}{detail}")

denied = [e for e in case["audit"] if e["decision"] == "deny"]
print(f"\n{len(case['audit'])} calls, {len(denied)} denied")

print("\n=== delegation ===")
for entry in TASK_LOG:
    usage = entry["usage"]
    print(
        f"{entry['agent']:<22} brief={entry['brief_chars']:>5} chars  "
        f"rounds={entry['iterations']}  "
        f"tokens={usage['input_tokens']}in/{usage['output_tokens']}out"
    )

print(f"\n=== case ===\nstate: {case['state']}\nactions: {case['actions']}")
leaked = [f for f in case["findings"] if re.search(r"\b\d{13,16}\b", json.dumps(f))]
print(f"findings containing an unmasked card number: {len(leaked)}")

=== audit ===
ok   order_investigation  lookup_order                
ok   order_investigation  get_shipment_events         
ok   billing_analysis     get_payment_events            (redact_card_numbers)
ok   billing_analysis     get_refund_status           
ok   order_investigation  report_finding              
ok   coordinator          Task                        
ok   billing_analysis     report_finding              
ok   coordinator          Task                        
ok   coordinator          escalate_to_human           

9 calls, 0 denied

=== delegation ===
order_investigation    brief=  361 chars  rounds=2  tokens=4104in/595out
billing_analysis       brief=  359 chars  rounds=2  tokens=3860in/659out

=== case ===
state: escalated
actions: [{'tool': 'escalate_to_human', 'order_id': 'A1009', 'id': 'ESC-BBB628'}]
findings containing an unmasked card number: 0


## Where this leaves us

The four layers, and what each one is actually for:

| Layer | Question it answers | Scope |
| --- | --- | --- |
| tool list | what can this agent even reach? | one agent |
| schema | are these arguments well-formed? | one tool |
| semantic validator | do the facts support this action? | one tool + the case |
| **hook** | **is this caller allowed to do this, now?** | **every call** |

They are not redundant. Each catches something the others structurally cannot,
and the ones that overlap do so on purpose — `create_return_authorization`
re-checks eligibility itself even though the validator already did, because the
only thing worse than a redundant check is a missing one.

Two things a hook layer buys that are easy to miss:

**Enforcement became testable.** `b4proof` runs the entire security model with no
model in the loop. "No sub-agent can write" went from a property you verify by
reading four tool lists to a single assertion that fails loudly in CI.

**Observation became free.** The audit trail is a by-product of the same
mechanism, not a separate logging effort someone has to remember to maintain —
and because `no_repeat_calls` reads it, it can't silently rot.

What is still open, if you want to keep going:

1. **No task budget.** `MAX_ITERATIONS` and `MAX_TASKS_PER_CASE` are guillotines
   — they cut the model off mid-thought. A real token budget tells the model how
   much it has left so it can pace itself and wrap up gracefully. That needs the
   larger models; notebook 8 flagged it and it's still open.
2. **Nothing evaluates.** Notebook 4 built a model-based grader and it has not
   been pointed at any of this. Every improvement since has been judged by
   reading transcripts, which does not scale and is not evidence.
3. **The audit is in memory.** `case["audit"]` dies with the kernel. Everything
   needed to persist it is already structured; nothing writes it down.